# Explore Activity Data 🏃

**Goal of this notebook:** before we design any dashboard screens, we need to see what
data Garmin Connect actually gives us for a run. This notebook:

1. Logs in to Garmin Connect (reusing the cached session token, just like `example.py`).
2. Finds today's activity (e.g. the interval workout you created in the Garmin Connect app).
3. Pulls the raw data for that activity: the summary, the lap/split data, and the
   "typed splits" (which show structured workout steps like warmup / interval / recovery).
4. Searches through that raw data for the running-dynamics fields we care about:
   **pace, cadence, ground contact time (GCT), vertical oscillation**, etc.
5. Saves the raw JSON to `your_data/` so we can refer back to real field names while
   building the dashboard (this folder is git-ignored, so your data never gets committed).

Nothing here builds any UI yet — it's purely "detective work" so the dashboard we build
next is based on real data, not guesses.

## 1. Connect to Garmin

We reuse the same login approach as `example.py`: read `EMAIL` / `PASSWORD` from the
`.env` file, and cache the session token in `~/.garminconnect` so we don't have to log
in every time we run this notebook.

In [1]:
import json
import os
import sys
from datetime import date
from pathlib import Path

# Let this notebook import the local `garminconnect` package no matter where
# Jupyter's working directory happens to be (repo root, notebooks/, etc).
REPO_ROOT = Path.cwd() if (Path.cwd() / "garminconnect").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

# Where we'll save raw JSON responses so we can inspect real field names later.
DATA_DIR = REPO_ROOT / "your_data"
DATA_DIR.mkdir(exist_ok=True)

In [2]:
def get_garmin_client() -> Garmin:
    """Log in to Garmin Connect, reusing a cached token if we have one."""
    tokenstore = Path(os.getenv("GARMINTOKENS", "~/.garminconnect")).expanduser()

    try:
        client = Garmin()
        client.login(str(tokenstore))
        print("✅ Logged in using saved tokens.")
        return client
    except GarminConnectTooManyRequestsError as err:
        raise SystemExit(f"Rate limited by Garmin, try again later: {err}") from err
    except (GarminConnectAuthenticationError, GarminConnectConnectionError):
        print("No valid saved tokens — logging in with email/password from .env ...")

    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")
    if not email or not password:
        raise SystemExit("Set EMAIL and PASSWORD in your .env file first.")

    client = Garmin(email=email, password=password)
    client.login(str(tokenstore))
    print(f"✅ Login successful. Tokens saved to: {tokenstore}")
    return client


api = get_garmin_client()
print("Logged in as:", api.get_full_name())

✅ Logged in using saved tokens.
Logged in as: Robert Sparks


In [3]:
import pandas as pd


def save_raw(name: str, data: object) -> Path:
    """Save a raw API response as pretty-printed JSON under your_data/, for reference."""
    out_path = DATA_DIR / f"{name}.json"
    out_path.write_text(json.dumps(data, indent=2, default=str))
    print(f"💾 Saved {name} -> {out_path.relative_to(REPO_ROOT)}")
    return out_path


def to_lap_frame(data: object) -> pd.DataFrame:
    """Turn a splits/typed-splits response into a per-lap pandas table.

    Garmin's split endpoints normally return a dict shaped like
    ``{"lapDTOs": [ {...one lap...}, {...next lap...} ]}``. We look for that key
    first, but fall back to scanning for *any* list-of-dicts in the response so
    this keeps working even if Garmin names the field differently for your
    account/activity type.
    """
    if isinstance(data, list):
        return pd.json_normalize(data)

    if isinstance(data, dict):
        if isinstance(data.get("lapDTOs"), list):
            return pd.json_normalize(data["lapDTOs"])
        for value in data.values():
            if isinstance(value, list) and value and isinstance(value[0], dict):
                return pd.json_normalize(value)

    print("⚠️ Could not find a list of laps automatically — inspect the raw JSON above.")
    return pd.DataFrame()


def find_fields(data: object, keywords: list[str], path: str = "") -> list[tuple[str, object]]:
    """Recursively search a nested dict/list for keys containing any of `keywords`.

    Returns a list of (path, example_value) tuples. Useful for hunting down fields
    like "pace", "cadence", "groundContact", "verticalOscillation", etc. without
    knowing the exact key name Garmin uses.
    """
    matches: list[tuple[str, object]] = []
    keywords_lower = [k.lower() for k in keywords]

    if isinstance(data, dict):
        for key, value in data.items():
            new_path = f"{path}.{key}" if path else key
            if any(kw in key.lower() for kw in keywords_lower):
                matches.append((new_path, value))
            matches.extend(find_fields(value, keywords, new_path))
    elif isinstance(data, list):
        for item in data[:1]:  # one example element is enough to see the shape
            matches.extend(find_fields(item, keywords, f"{path}[0]"))

    return matches

## 2. Find today's activity

`get_activities_by_date(start, end)` returns every activity recorded between two dates
(inclusive). We ask for just today, which will include the workout you created in the
Garmin Connect app once it's synced from your watch.

In [ ]:
today = date.today().isoformat()
todays_activities = api.get_activities_by_date(today, today)

print(f"Found {len(todays_activities)} activities today ({today}).\n")

for activity in todays_activities:
    print(
        f"- id={activity['activityId']} | "
        f"'{activity['activityName']}' | "
        f"type={activity['activityType']['typeKey']} | "
        f"start={activity['startTimeLocal']} | "
        f"distance={activity.get('distance', 0) / 1000:.2f} km | "
        f"duration={activity.get('duration', 0) / 60:.1f} min"
    )

if not todays_activities:
    print(
        "No activities found for today yet. If you just finished the workout, "
        "make sure it has synced to Garmin Connect (via the app or a Bluetooth sync), "
        "then re-run this cell."
    )

Found 1 activities today (2026-07-02).

- id=23450669285 | 'County Dublin - intervals' | type=running | start=2026-07-02 08:26:57 | distance=10.88 km | duration=55.5 min


## 3. Pull the full summary for the latest activity

Garmin returns activities newest-first, so `todays_activities[0]` is today's most
recent activity. We fetch its full summary with `get_activity()`, which includes
overall stats (distance, average pace, average cadence, etc. — the whole-run
averages, not per-interval).

If you have more than one activity today, change `ACTIVITY_INDEX` below to pick a
different one (0 = most recent).

In [5]:
ACTIVITY_INDEX = 0  # 0 = today's most recent activity

if not todays_activities:
    raise SystemExit("No activities today yet — nothing to explore.")

activity_id = todays_activities[ACTIVITY_INDEX]["activityId"]
activity_summary = api.get_activity(activity_id)
save_raw("latest_activity_summary", activity_summary)

print(f"Exploring activity {activity_id}: '{activity_summary.get('activityName')}'\n")
print(json.dumps(activity_summary, indent=2, default=str))

💾 Saved latest_activity_summary -> your_data/latest_activity_summary.json
Exploring activity 23450669285: 'County Dublin - intervals'

{
  "activityId": 23450669285,
  "activityUUID": {
    "uuid": "e2008f73-b7a5-46c2-a7a4-a38c49e81a23"
  },
  "activityName": "County Dublin - intervals",
  "userProfileId": 69672142,
  "isMultiSportParent": false,
  "activityTypeDTO": {
    "typeId": 1,
    "typeKey": "running",
    "parentTypeId": 17,
    "isHidden": false,
    "restricted": false,
    "trimmable": true
  },
  "eventTypeDTO": {
    "typeId": 2,
    "typeKey": "recreation",
    "sortOrder": 4
  },
  "accessControlRuleDTO": {
    "typeId": 3,
    "typeKey": "subscribers"
  },
  "timeZoneUnitDTO": {
    "unitId": 159,
    "unitKey": "Europe/London",
    "factor": 0.0,
    "timeZone": "Europe/London"
  },
  "metadataDTO": {
    "isOriginal": true,
    "deviceApplicationInstallationId": 1011746,
    "agentApplicationInstallationId": null,
    "agentString": null,
    "fileFormat": {
      "

## 4. Look at the lap/split data (this is where intervals live)

`get_activity_splits()` returns one entry per lap. For a workout you built in the
Garmin Connect app (warmup / interval / recovery / cooldown), each lap usually
corresponds to one step of that workout — this is how we'll identify "the interval
sections" for the dashboard.

Let's dump the raw JSON first, then load it into a pandas table so it's easier to
scan for useful columns (pace, cadence, GCT, vertical oscillation, heart rate...).

In [6]:
activity_splits = api.get_activity_splits(activity_id)
save_raw("latest_activity_splits", activity_splits)
print(json.dumps(activity_splits, indent=2, default=str))

💾 Saved latest_activity_splits -> your_data/latest_activity_splits.json
{
  "activityId": 23450669285,
  "lapDTOs": [
    {
      "startTimeGMT": "2026-07-02T07:26:57.0",
      "startLatitude": 53.484583077952266,
      "startLongitude": -6.147782485932112,
      "distance": 1000.0,
      "duration": 355.265,
      "movingDuration": 347.0,
      "elapsedDuration": 355.265,
      "elevationGain": 8.0,
      "elevationLoss": 8.0,
      "maxElevation": 19.6,
      "minElevation": 11.8,
      "averageSpeed": 2.815000057220459,
      "averageMovingSpeed": 2.881844380403458,
      "maxSpeed": 3.302999973297119,
      "calories": 70.0,
      "bmrCalories": 9.0,
      "averageHR": 142.0,
      "maxHR": 166.0,
      "averageRunCadence": 170.71875,
      "maxRunCadence": 180.0,
      "averagePower": 306.0,
      "maxPower": 461.0,
      "minPower": 0.0,
      "normalizedPower": 320.0,
      "totalWork": 26.014195636297682,
      "groundContactTime": 260.29998779296875,
      "strideLength": 97.9

In [7]:
laps_df = to_lap_frame(activity_splits)

# Show only columns that actually have data for at least one lap, so the table
# isn't cluttered with empty/irrelevant fields.
laps_df = laps_df.dropna(axis=1, how="all")
laps_df

,startTimeGMT,startLatitude,startLongitude,distance,duration,movingDuration,elapsedDuration,elevationGain,elevationLoss,maxElevation,...,maxVerticalSpeed,avgGradeAdjustedSpeed,lapIndex,wktStepIndex,lengthDTOs,wktIndex,connectIQMeasurement,intensityType,messageIndex,directWorkoutComplianceScore
0,2026-07-02T07:26:57.0,53.484583,-6.147782,1000.00,355.265,347.000,355.265,8.0,8.0,19.6,...,0.400002,2.854,1,0,[],0,[],WARMUP,0,NaN
1,2026-07-02T07:32:53.0,53.484642,-6.157721,1000.00,311.634,311.634,311.634,5.0,9.0,15.0,...,0.400001,3.209,2,0,[],0,[],WARMUP,1,NaN
2,2026-07-02T07:38:05.0,53.486654,-6.169541,93.24,27.384,26.000,27.384,1.0,0.0,11.8,...,0.200001,3.469,3,0,[],0,[],WARMUP,2,NaN
3,2026-07-02T07:38:31.0,53.487047,-6.170604,123.92,90.000,90.000,90.000,0.0,1.0,11.6,...,0.200001,1.416,4,1,[],0,[],RECOVERY,3,100.0
4,2026-07-02T07:40:01.0,53.487689,-6.171996,1000.00,254.866,254.866,254.866,5.0,1.0,13.8,...,0.600000,3.909,5,2,[],0,[],ACTIVE,4,86.0
5,2026-07-02T07:44:17.0,53.488260,-6.162477,1000.00,253.074,253.000,253.074,5.0,8.0,14.6,...,0.200001,3.935,6,2,[],0,[],ACTIVE,5,87.0
6,2026-07-02T07:48:30.0,53.483757,-6.163446,112.65,90.000,83.000,90.000,0.0,5.0,10.2,...,0.400000,1.318,7,1,[],0,[],RECOVERY,6,100.0
7,2026-07-02T07:49:59.0,53.483853,-6.164876,1000.00,255.212,255.212,255.212,12.0,7.0,11.2,...,0.800000,3.868,8,2,[],0,[],ACTIVE,7,82.0
8,2026-07-02T07:54:15.0,53.489442,-6.172334,1000.00,254.292,254.292,254.292,4.0,2.0,13.2,...,0.200001,3.950,9,2,[],0,[],ACTIVE,8,83.0
9,2026-07-02T07:58:30.0,53.487093,-6.158684,119.60,90.000,89.000,90.000,0.0,1.0,11.2,...,0.200000,1.387,10,1,[],0,[],RECOVERY,9,100.0


## 5. Typed splits — structured workout step labels

Because this activity was created as a structured workout, `get_activity_typed_splits()`
should tell us the *type* of each split (e.g. `WARMUP`, `INTERVAL`, `RECOVERY`,
`COOLDOWN`), which is exactly what we need to group laps into "the interval section"
vs "the recovery section" vs "the warmup" on the dashboard.

In [8]:
activity_typed_splits = api.get_activity_typed_splits(activity_id)
save_raw("latest_activity_typed_splits", activity_typed_splits)
print(json.dumps(activity_typed_splits, indent=2, default=str))

💾 Saved latest_activity_typed_splits -> your_data/latest_activity_typed_splits.json
{
  "activityId": 23450669285,
  "activityUUID": {
    "uuid": "e2008f73-b7a5-46c2-a7a4-a38c49e81a23"
  },
  "splits": [
    {
      "startTimeLocal": "2026-07-02T08:26:57.0",
      "startTimeGMT": "2026-07-02T07:26:57.0",
      "distance": 0.0,
      "duration": 9.001,
      "movingDuration": 0.0,
      "elapsedDuration": 9.001,
      "elevationGain": 0.0,
      "elevationLoss": 0.0,
      "averageSpeed": 0.0,
      "maxSpeed": 0.0,
      "calories": 1.0,
      "bmrCalories": 0.0,
      "averageHR": 94.0,
      "maxHR": 95.0,
      "averageRunCadence": 19.65625,
      "maxRunCadence": 177.0,
      "averagePower": 2.0,
      "maxPower": 14.0,
      "strideLength": 0.0,
      "totalExerciseReps": 0,
      "endLatitude": 53.484583077952266,
      "endLongitude": -6.147782485932112,
      "avgVerticalSpeed": 0.0,
      "avgGradeAdjustedSpeed": 0.0,
      "avgElapsedDurationVerticalSpeed": 0.028571401323590

In [9]:
typed_laps_df = to_lap_frame(activity_typed_splits)
typed_laps_df = typed_laps_df.dropna(axis=1, how="all")
typed_laps_df

,startTimeLocal,startTimeGMT,distance,duration,movingDuration,elapsedDuration,elevationGain,elevationLoss,averageSpeed,maxSpeed,...,startElevation,startLatitude,startLongitude,averageMovingSpeed,normalizedPower,groundContactTime,verticalOscillation,verticalRatio,lapIndexes,avgStepLength
0,2026-07-02T08:26:57.0,2026-07-02T07:26:57.0,0.00,9.001,0.000,9.001,0.0,0.0,0.000,0.000,...,29.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-07-02T08:26:57.0,2026-07-02T07:26:57.0,2093.24,694.297,683.000,694.297,14.0,16.0,3.015,4.227,...,29.6,53.484583,-6.147782,3.064773,341.0,253.600006,8.42,8.22,"[1, 2, 3]",1.0294
2,2026-07-02T08:27:06.0,2026-07-02T07:27:06.0,2115.10,699.000,697.000,699.000,14.0,16.0,3.026,4.227,...,30.0,53.484560,-6.147795,3.034577,341.0,253.600006,8.41,8.22,NaN,1.0276
3,2026-07-02T08:38:31.0,2026-07-02T07:38:31.0,123.92,90.000,89.000,90.000,0.0,1.0,1.377,3.014,...,11.4,53.487047,-6.170604,1.392360,239.0,268.000000,5.95,8.63,[4],0.6950
4,2026-07-02T08:38:45.0,2026-07-02T07:38:45.0,98.94,75.000,74.000,75.000,0.0,1.0,1.319,1.446,...,11.4,53.487210,-6.170742,1.337027,156.0,NaN,5.78,8.57,NaN,0.6749
5,2026-07-02T08:40:00.0,2026-07-02T07:40:00.0,2031.67,524.000,523.000,524.000,9.0,9.0,3.877,4.143,...,9.4,53.487680,-6.171952,3.884646,385.0,218.000000,8.20,6.72,NaN,1.2232
6,2026-07-02T08:40:01.0,2026-07-02T07:40:01.0,2000.00,507.938,507.938,507.938,9.0,9.0,3.937,4.143,...,9.2,53.487689,-6.171996,3.937489,384.0,217.600006,8.21,6.70,"[5, 6]",1.2276
7,2026-07-02T08:48:30.0,2026-07-02T07:48:30.0,112.65,90.000,82.000,89.997,0.0,5.0,1.252,3.872,...,10.2,53.483757,-6.163446,1.373781,283.0,232.300003,6.06,8.18,[7],0.7862
8,2026-07-02T08:48:44.0,2026-07-02T07:48:44.0,44.14,48.000,45.000,48.000,0.0,2.0,0.920,1.474,...,10.0,53.483683,-6.163794,0.980889,212.0,NaN,5.74,8.60,NaN,0.6720
9,2026-07-02T08:49:32.0,2026-07-02T07:49:32.0,3.85,3.000,0.000,3.000,0.0,0.0,1.283,0.000,...,6.8,53.483634,-6.164428,NaN,50.0,NaN,NaN,NaN,NaN,NaN


## 6. Hunt for the running-dynamics fields we actually care about

Rather than guessing exact Garmin field names, let's search everything we've pulled so
far for keys that look related to **pace, cadence, ground contact time (GCT),
vertical oscillation, stride length**, and **heart rate**. This tells us exactly which
field to use for each stat on the dashboard.

In [10]:
KEYWORDS = [
    "pace",
    "speed",
    "cadence",
    "groundcontact",
    "ground_contact",
    "verticaloscillation",
    "verticalratio",
    "stride",
    "heartrate",
    "hr",
]

all_sources = {
    "activity_summary": activity_summary,
    "activity_splits": activity_splits,
    "activity_typed_splits": activity_typed_splits,
}

for source_name, source_data in all_sources.items():
    print(f"\n=== {source_name} ===")
    hits = find_fields(source_data, KEYWORDS)
    if not hits:
        print("  (no matching fields found)")
    for field_path, example_value in hits:
        print(f"  {field_path} = {example_value!r}")


=== activity_summary ===
  metadataDTO.hasHrTimeInZones = True
  summaryDTO.averageSpeed = 3.2679998874664307
  summaryDTO.averageMovingSpeed = 3.2868925855001545
  summaryDTO.maxSpeed = 4.2270002365112305
  summaryDTO.averageHR = 169.0
  summaryDTO.maxHR = 202.0
  summaryDTO.minHR = 92.0
  summaryDTO.averageRunCadence = 175.0
  summaryDTO.maxRunCadence = 197.0
  summaryDTO.groundContactTime = 233.39999389648438
  summaryDTO.strideLength = 110.85000000000001
  summaryDTO.verticalOscillation = 8.040000152587892
  summaryDTO.verticalRatio = 7.429999828338623
  summaryDTO.maxVerticalSpeed = 1.7999997138977053
  summaryDTO.avgGradeAdjustedSpeed = 3.2739999294281006
  splitSummaries[0].averageSpeed = 0.32100000977516174
  splitSummaries[0].maxSpeed = 0.0
  splitSummaries[0].averageHR = 105.0
  splitSummaries[0].maxHR = 138.0
  splitSummaries[0].averageRunCadence = 19.65625
  splitSummaries[0].maxRunCadence = 177.0
  splitSummaries[0].strideLength = 0.0
  splitSummaries[0].avgVerticalSpeed 

## Next steps

Now that we can see the real field names Garmin gives us, here's what's next:

1. **Review the output above** — check the `lapDTOs` tables in sections 4 & 5, and the
   field hits in section 6. Confirm which lap corresponds to which part of your workout
   (warmup / interval / recovery / cooldown), and which exact field names hold pace,
   cadence, GCT, and vertical oscillation for your watch model (these can vary slightly
   between Garmin devices).
2. **Check `your_data/`** — the raw JSON files saved by this notebook
   (`latest_activity_summary.json`, `latest_activity_splits.json`,
   `latest_activity_typed_splits.json`) are there for reference any time, without
   needing to re-run API calls.
3. Once we've confirmed the fields, we'll design the dashboard: a small local web app
   (Streamlit is a great beginner-friendly choice for a Python data dashboard) that
   shows your latest run with a summary card up top and a per-interval breakdown table
   below, using the exact field names we found here.